# IPL Ball-by-Ball Analysis
Dataset: `archive/IPL.csv` — 278,205 deliveries across 18 IPL seasons (2007/08 – 2025)

In [ ]:
import pandas as pd
import numpy as np

## Load Data

In [ ]:
df = pd.read_csv('../archive/IPL.csv', index_col=0, low_memory=False)

# Fix mixed-type columns flagged by pandas
mixed_cols = ['team_reviewed', 'review_decision', 'umpire', 'umpires_call',
              'gender', 'result_type', 'method', 'balls_per_over', 'stage']
df[mixed_cols] = df[mixed_cols].astype(str).replace('nan', pd.NA)

print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()

## Schema & Nulls

In [ ]:
print('--- dtypes ---')
print(df.dtypes)
print('\n--- null counts (non-zero only) ---')
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False))

## Data Distribution

In [ ]:
# Deliveries per season
print('--- Deliveries per season ---')
print(df['season'].value_counts().sort_index())

In [ ]:
# Numeric summary for key batting/bowling columns
numeric_cols = ['runs_batter', 'runs_extras', 'runs_total', 'over', 'innings']
df[numeric_cols].describe().round(3)

In [ ]:
# Runs distribution
print('--- runs_batter value counts ---')
print(df['runs_batter'].value_counts().sort_index())

print('\n--- Boundary rate (4s + 6s) ---')
boundaries = df['runs_batter'].isin([4, 6]).sum()
valid_balls = df['valid_ball'].sum()
print(f'{boundaries:,} boundaries in {valid_balls:,} valid deliveries ({boundaries/valid_balls:.1%})')

In [ ]:
# Wicket kinds
print('--- Wicket kinds ---')
print(df['wicket_kind'].value_counts())

In [ ]:
# Extra types
print('--- Extra types ---')
print(df['extra_type'].value_counts())

In [ ]:
# Teams
print(f'--- {df["batting_team"].nunique()} unique teams ---')
print(df['batting_team'].value_counts())

In [ ]:
# Top venues
print(f'--- {df["venue"].nunique()} unique venues (top 10) ---')
print(df['venue'].value_counts().head(10))

## Per-(match, innings) Aggregation

Goal: one row per team's batting innings — two rows per match. Supports both team-total prediction and per-player supervision.

Row schema
- `match_id`, `innings` (1 or 2), `season`, `venue`, `city`
- `batting_team`, `bowling_team`, `won_toss` (bool), `toss_decision`
- `players`       — full XI of the batting team (alphabetical), as `a|b|...`
- `bowlers`       — full XI of the bowling team (alphabetical)
- `runs_scored`   — per-batter runs in this innings, pipe-separated, aligned with `players`
- `did_bat`       — 0/1 pipe-separated flag, aligned with `players` (0 for XI members who didn't face a ball)
- `runs_conceded` — per-bowler runs conceded in this innings, aligned with `bowlers`
- `did_bowl`      — 0/1 pipe-separated flag, aligned with `bowlers`
- `team_runs`     — total runs scored by the batting team this innings
- `team_won`      — did this batting team win the match

Rain-affected matches (D/L applied, no result) are dropped. Team names are mapped to acronyms via `TEAM_MAP`.

The full XI for a team in a given match is reconstructed as the union of everyone who batted, bowled, or fielded for that side across both innings.

In [ ]:
TEAM_MAP = {
    'Chennai Super Kings': 'CSK',
    'Mumbai Indians': 'MI',
    'Royal Challengers Bangalore': 'RCB',
    'Royal Challengers Bengaluru': 'RCB',
    'Kolkata Knight Riders': 'KKR',
    'Delhi Daredevils': 'DC',
    'Delhi Capitals': 'DC',
    'Kings XI Punjab': 'PBKS',
    'Punjab Kings': 'PBKS',
    'Rajasthan Royals': 'RR',
    'Sunrisers Hyderabad': 'SRH',
    'Gujarat Titans': 'GT',
    'Lucknow Super Giants': 'LSG',
    'Deccan Chargers': 'DCH',
    'Pune Warriors': 'PWI',
    'Rising Pune Supergiant': 'RPSG',
    'Rising Pune Supergiants': 'RPSG',
    'Gujarat Lions': 'GL',
    'Kochi Tuskers Kerala': 'KTK',
}

df_reg = df[df['innings'].isin([1, 2])].copy()

# Drop rain-affected matches: D/L applied or abandoned (no result)
flags = df_reg.groupby('match_id').agg(
    method=('method', lambda s: s.dropna().iloc[0] if s.dropna().size else None),
    result_type=('result_type', lambda s: s.dropna().iloc[0] if s.dropna().size else None),
)
drop_ids = set(flags[(flags['method'] == 'D/L') | (flags['result_type'] == 'no result')].index)
df_reg = df_reg[~df_reg['match_id'].isin(drop_ids)]
print(f'Dropped {len(drop_ids)} rain-affected matches')


def split_fielders(s):
    if pd.isna(s):
        return []
    return [x.strip() for x in str(s).split(',') if x.strip()]


def _innings_row(match_id, season, venue, city, toss_winner_code, toss_decision,
                 winner_raw, batting_raw, bowling_raw, batting_squad, bowling_squad, inn):
    batting_code = TEAM_MAP.get(batting_raw, batting_raw)
    bowling_code = TEAM_MAP.get(bowling_raw, bowling_raw)

    # Per-batter runs scored (only players who faced a ball appear here)
    batter_runs = inn.groupby('batter')['runs_batter'].sum().astype(int).to_dict()
    batted = set(inn['batter'].dropna()) | set(inn['non_striker'].dropna())

    # Per-bowler runs conceded (only players who bowled a ball appear here)
    bowler_runs = inn.groupby('bowler')['runs_bowler'].sum().astype(int).to_dict()
    bowled = set(inn['bowler'].dropna())

    return {
        'match_id':      match_id,
        'innings':       int(inn['innings'].iloc[0]),
        'season':        season,
        'venue':         venue,
        'city':          city,
        'batting_team':  batting_code,
        'bowling_team':  bowling_code,
        'won_toss':      int(toss_winner_code == batting_code),
        'toss_decision': toss_decision,
        'players':       '|'.join(batting_squad),
        'bowlers':       '|'.join(bowling_squad),
        'runs_scored':   '|'.join(str(batter_runs.get(p, 0)) for p in batting_squad),
        'did_bat':       '|'.join('1' if p in batted else '0' for p in batting_squad),
        'runs_conceded': '|'.join(str(bowler_runs.get(p, 0)) for p in bowling_squad),
        'did_bowl':      '|'.join('1' if p in bowled else '0' for p in bowling_squad),
        'team_runs':     int(inn['runs_total'].sum()),
        'team_won':      int(winner_raw == batting_raw),
    }


rows = []
for mid, g in df_reg.groupby('match_id', sort=False):
    inn1 = g[g['innings'] == 1]
    inn2 = g[g['innings'] == 2]
    if inn1.empty or inn2.empty:
        continue

    t1 = inn1['batting_team'].iloc[0]
    t2 = inn2['batting_team'].iloc[0]
    winner_raw = g['match_won_by'].iloc[0]
    toss_winner_raw = g['toss_winner'].iloc[0]
    toss_winner_code = TEAM_MAP.get(toss_winner_raw, toss_winner_raw)
    toss_decision = g['toss_decision'].iloc[0]
    season = g['season'].iloc[0]
    venue = g['venue'].iloc[0]
    city = g['city'].iloc[0]

    # Full XI for each side: union of batters + non_strikers in own innings
    # and bowlers + fielders in the opposing innings.
    t1_squad = sorted(
        set(inn1['batter'].dropna()) | set(inn1['non_striker'].dropna())
        | set(inn2['bowler'].dropna())
        | {f for lst in inn2['fielders'].map(split_fielders) for f in lst}
    )
    t2_squad = sorted(
        set(inn2['batter'].dropna()) | set(inn2['non_striker'].dropna())
        | set(inn1['bowler'].dropna())
        | {f for lst in inn1['fielders'].map(split_fielders) for f in lst}
    )

    rows.append(_innings_row(
        mid, season, venue, city, toss_winner_code, toss_decision,
        winner_raw, t1, t2, t1_squad, t2_squad, inn1,
    ))
    rows.append(_innings_row(
        mid, season, venue, city, toss_winner_code, toss_decision,
        winner_raw, t2, t1, t2_squad, t1_squad, inn2,
    ))

matches = pd.DataFrame(rows)
print(f"{len(matches):,} rows = {len(matches) // 2:,} matches × 2 innings")
print(f"team_won rate (should be ~0.5): {matches['team_won'].mean():.3f}")
matches.head()

In [ ]:
# Sanity checks: squad sizes and alignment of per-player columns.
sizes = pd.DataFrame({
    'players':        matches['players'].str.split('|').str.len(),
    'bowlers':        matches['bowlers'].str.split('|').str.len(),
    'runs_scored':    matches['runs_scored'].str.split('|').str.len(),
    'did_bat':        matches['did_bat'].str.split('|').str.len(),
    'runs_conceded':  matches['runs_conceded'].str.split('|').str.len(),
    'did_bowl':       matches['did_bowl'].str.split('|').str.len(),
})
print('Squad/aligned-column sizes:')
print(sizes.describe().round(2))

# Column-pair length mismatches must be zero
assert (sizes['players'] == sizes['runs_scored']).all()
assert (sizes['players'] == sizes['did_bat']).all()
assert (sizes['bowlers'] == sizes['runs_conceded']).all()
assert (sizes['bowlers'] == sizes['did_bowl']).all()
print('\nAll per-player columns aligned with player lists ✓')

# Spot-check: batter runs should sum to less than team_runs (difference = extras)
sample = matches.iloc[0]
bat_sum = sum(int(r) for r in sample['runs_scored'].split('|'))
print(f"\nrow 0 · team_runs={sample['team_runs']}, "
      f"sum(runs_scored)={bat_sum}, extras={sample['team_runs'] - bat_sum}")

# Fraction of XI members who actually bat / bowl
did_bat_rate  = matches['did_bat'].str.split('|').apply(lambda xs: sum(int(x) for x in xs) / len(xs)).mean()
did_bowl_rate = matches['did_bowl'].str.split('|').apply(lambda xs: sum(int(x) for x in xs) / len(xs)).mean()
print(f"mean did_bat  rate per XI: {did_bat_rate:.2f}  (≈ 0.7–0.9 expected)")
print(f"mean did_bowl rate per XI: {did_bowl_rate:.2f}  (≈ 0.4–0.6 expected — only 5–6 of 11 bowl)")

In [ ]:
out_path = '../archive/matches_aggregated.csv'
matches.to_csv(out_path, index=False)
print(f'Wrote {out_path}')

## Per-Player Career Stats

Career batting and bowling stats for every player in the ball-by-ball data. Fed as extra per-player features into the Stage 1 embedding model (Option B). Scope matches `matches_aggregated.csv` — innings 1 and 2, rain-affected matches excluded.

Columns (redundant counts dropped — strike_rate subsumes balls_faced, economy subsumes runs_conceded):
- `runs`, `strike_rate` — batting volume + rate
- `wickets`, `balls_bowled`, `economy` — bowling volume + workload + rate

Players who only batted get 0s in the bowling columns, and vice versa.

**Leakage note:** these are career totals through 2025. Using them directly in Stage 1 for a 2012 match uses future information. Fine as a starting point; for a clean pipeline regenerate as-of each match date.

In [ ]:
# Batting stats: runs (volume) + strike rate (rate)
bat_stats = (df_reg.groupby('batter', sort=False)
             .agg(runs=('runs_batter', 'sum'),
                  _balls=('balls_faced', 'sum'))
             .reset_index()
             .rename(columns={'batter': 'player'}))
bat_stats['strike_rate'] = (bat_stats['runs'] * 100.0 / bat_stats['_balls']).where(
    bat_stats['_balls'] > 0, 0.0)
bat_stats = bat_stats.drop(columns='_balls')

# Bowling stats: wickets (volume) + balls bowled (workload) + economy (rate)
bowl_stats = (df_reg.groupby('bowler', sort=False)
              .agg(wickets=('bowler_wicket', 'sum'),
                   _runs=('runs_bowler', 'sum'),
                   balls_bowled=('valid_ball', 'sum'))
              .reset_index()
              .rename(columns={'bowler': 'player'}))
bowl_stats['economy'] = (bowl_stats['_runs'] * 6.0 / bowl_stats['balls_bowled']).where(
    bowl_stats['balls_bowled'] > 0, 0.0)
bowl_stats = bowl_stats.drop(columns='_runs')

player_stats = (bat_stats.merge(bowl_stats, on='player', how='outer')
                .fillna(0)
                .sort_values('player')
                .reset_index(drop=True))

for c in ['runs', 'wickets', 'balls_bowled']:
    player_stats[c] = player_stats[c].astype(int)

player_stats = player_stats[['player',
                             'runs', 'strike_rate',
                             'wickets', 'balls_bowled', 'economy']]

print(f'{len(player_stats):,} players')
print('\nTop 5 by runs:')
print(player_stats.sort_values('runs', ascending=False).head(5).to_string(index=False))
print('\nTop 5 by wickets:')
print(player_stats.sort_values('wickets', ascending=False).head(5).to_string(index=False))

In [ ]:
stats_path = '../archive/player_stats.csv'
player_stats.to_csv(stats_path, index=False)
print(f'Wrote {stats_path}')